# Homework 10a: Modeling, Linear Regression

Fits `LinearRegression` on a synthetic finance factor dataset (market excess return, size,
value, momentum, versus an asset's excess return), diagnoses the residuals, then adds a
momentum-squared term the data was generated with, to see whether the diagnostics improve.

In [1]:
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.linear_model import LinearRegression
from sklearn.metrics import r2_score, mean_squared_error
from sklearn.model_selection import train_test_split
import scipy.stats as st

sns.set_theme()
np.random.seed(7)

## 1. Synthetic Data

Finance-flavored factors and an asset's excess return, generated with a genuine quadratic
momentum effect baked in (`beta_mom2 * momentum**2`), so a plain linear fit has something real
to be mis-specified about.

In [2]:
n = 200
dates = pd.bdate_range(start="2024-02-01", periods=n)
mkt_excess = np.random.normal(0, 0.011, size=n)
size = np.random.normal(0, 0.008, size=n)
value = np.random.normal(0, 0.009, size=n)
momentum = np.random.normal(0, 0.006, size=n)

beta0, beta_mkt, beta_size, beta_value, beta_mom, beta_mom2 = 0.0001, 0.9, 0.25, -0.15, 0.35, 3.5
noise_scale = 0.0035 + 0.5 * np.abs(mkt_excess)
eps = np.random.normal(0, noise_scale)
asset_excess = (
    beta0 + beta_mkt * mkt_excess + beta_size * size + beta_value * value + beta_mom * momentum
    + beta_mom2 * (momentum ** 2) + eps
)
df = pd.DataFrame({
    'date': dates, 'mkt_excess': mkt_excess, 'size': size, 'value': value,
    'momentum': momentum, 'asset_excess': asset_excess,
})
df.head()

,date,mkt_excess,size,value,momentum,asset_excess
0,2024-02-01,0.018596,-0.013467,-0.000540,0.000141,0.014832
1,2024-02-02,-0.005125,0.008120,0.007089,-0.006840,-0.008599
2,2024-02-05,0.000361,-0.011532,-0.012885,-0.011216,-0.009577
3,2024-02-06,0.004483,-0.010749,0.003905,-0.010685,0.000033
4,2024-02-07,-0.008678,-0.002893,0.001250,0.002779,-0.005750


## 2. Baseline Model

80/20 train/test split, no shuffling since this is time-ordered data and shuffling would let
the model train on the future and test on the past.

In [3]:
X = df[['mkt_excess', 'size', 'value', 'momentum']]
y = df['asset_excess']
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, shuffle=False)

lr = LinearRegression().fit(X_train, y_train)
y_pred = lr.predict(X_test)
r2 = r2_score(y_test, y_pred)
rmse = np.sqrt(mean_squared_error(y_test, y_pred))
print(f'Baseline   R2={r2:.4f}  RMSE={rmse:.6f}')
print('Coefficients:', dict(zip(X.columns, lr.coef_.round(4))))

Baseline   R2=0.3677  RMSE=0.008470
Coefficients: {'mkt_excess': np.float64(0.7785), 'size': np.float64(0.2311), 'value': np.float64(-0.2099), 'momentum': np.float64(0.1815)}


## 3. Residual Diagnostics

In [4]:
resid = y_test - y_pred
fitted = y_pred

fig, axes = plt.subplots(1, 3, figsize=(15, 4))
axes[0].scatter(fitted, resid)
axes[0].axhline(0, ls='--', color='black')
axes[0].set_xlabel('fitted'); axes[0].set_ylabel('residual')
axes[0].set_title('Residuals vs Fitted')

axes[1].hist(resid, bins=20)
axes[1].set_title('Residual Histogram')

st.probplot(resid, dist='norm', plot=axes[2])
axes[2].set_title('QQ Plot')

fig.tight_layout()
fig.savefig('residual_diagnostics_baseline.png', dpi=110)
plt.close(fig)
print('Saved residual_diagnostics_baseline.png')

Saved residual_diagnostics_baseline.png


In [5]:
fig, ax = plt.subplots(figsize=(6, 4))
ax.scatter(df['momentum'].iloc[-len(resid):], resid)
ax.axhline(0, ls='--', color='black')
ax.set_xlabel('momentum'); ax.set_ylabel('residual')
ax.set_title('Residuals vs momentum (a key predictor)')
fig.tight_layout()
fig.savefig('residuals_vs_momentum.png', dpi=110)
plt.close(fig)

fig, ax = plt.subplots(figsize=(6, 4))
ax.scatter(resid.to_numpy()[:-1], resid.to_numpy()[1:])
ax.axhline(0, ls='--', color='black'); ax.axvline(0, ls='--', color='black')
ax.set_xlabel('residual (t)'); ax.set_ylabel('residual (t+1)')
ax.set_title('Residual lag-1 (independence check)')
fig.tight_layout()
fig.savefig('residual_lag1.png', dpi=110)
plt.close(fig)

lag1_corr = np.corrcoef(resid.to_numpy()[:-1], resid.to_numpy()[1:])[0, 1]
shapiro_stat, shapiro_p = st.shapiro(resid)
hetero_corr = np.corrcoef(resid.abs().to_numpy(), X_test['mkt_excess'].abs().to_numpy())[0, 1]
print(f'Residual lag-1 autocorrelation: {lag1_corr:.3f}')
print(f'Shapiro-Wilk normality test: stat={shapiro_stat:.4f}, p={shapiro_p:.4f}')
print(f'Correlation of |residual| with |mkt_excess| (heteroscedasticity check): {hetero_corr:.3f}')

Residual lag-1 autocorrelation: 0.260
Shapiro-Wilk normality test: stat=0.9644, p=0.2355
Correlation of |residual| with |mkt_excess| (heteroscedasticity check): 0.566


## 4. Add the Transformed Feature: `momentum_sq`

In [6]:
df['momentum_sq'] = df['momentum'] ** 2
X2 = df[['mkt_excess', 'size', 'value', 'momentum', 'momentum_sq']]
X2_train, X2_test = X2.iloc[:len(X_train)], X2.iloc[len(X_train):]

lr2 = LinearRegression().fit(X2_train, y_train)
y_pred2 = lr2.predict(X2_test)
r2_2 = r2_score(y_test, y_pred2)
rmse_2 = np.sqrt(mean_squared_error(y_test, y_pred2))
print(f'With x^2   R2={r2_2:.4f}  RMSE={rmse_2:.6f}')
print('Coefficients:', dict(zip(X2.columns, lr2.coef_.round(4))))

resid2 = y_test - y_pred2
fig, axes = plt.subplots(1, 3, figsize=(15, 4))
axes[0].scatter(y_pred2, resid2)
axes[0].axhline(0, ls='--', color='black')
axes[0].set_title('Residuals vs Fitted (with x^2)')
axes[1].hist(resid2, bins=20)
axes[1].set_title('Residual Histogram (with x^2)')
st.probplot(resid2, dist='norm', plot=axes[2])
axes[2].set_title('QQ Plot (with x^2)')
fig.tight_layout()
fig.savefig('residual_diagnostics_with_x2.png', dpi=110)
plt.close(fig)
print('Saved residual_diagnostics_with_x2.png')

With x^2   R2=0.3681  RMSE=0.008467
Coefficients: {'mkt_excess': np.float64(0.7789), 'size': np.float64(0.2257), 'value': np.float64(-0.2012), 'momentum': np.float64(0.2111), 'momentum_sq': np.float64(20.2317)}


Saved residual_diagnostics_with_x2.png


Adding `momentum_sq` is still ordinary linear regression: "linear" describes the model's
linearity in its coefficients, not in the raw predictors. `y = b0 + b1*x + b2*x^2` is linear in
`b0`, `b1`, `b2`, even though the fitted curve bends. `LinearRegression` doesn't need to know
`momentum_sq` came from squaring another column, it's just another number in the design matrix.

## 5. Interpretation

**Linearity.** The residuals-vs-fitted plot for the baseline model doesn't show an obvious
curve, which is a little surprising given the data was generated with a real
`momentum_sq` effect (`beta_mom2 = 3.5`). Adding `momentum_sq` back in barely moves R2
(0.3677 to 0.3681) or RMSE (0.008470 to 0.008467) on this 40-row test set: the quadratic
effect is real in the data-generating process, but too small relative to the noise for this
particular holdout to reward modeling it. That's a useful reminder on its own: a diagnostic
plot not showing an obvious pattern doesn't prove the true relationship is linear, only that
the deviation isn't large enough to see by eye at this sample size and noise level.

**Homoscedasticity.** This one does show a real violation, and it's checked directly rather
than just eyeballed: `|residual|` correlates with `|mkt_excess|` in the test set. The data was
generated with `noise_scale = 0.0035 + 0.5 * |mkt_excess|` on purpose, so the residual spread
should widen on days when the market factor is large, and the correlation confirms it does.
Practically, that means the model's uncertainty isn't constant across days: prediction
intervals should be wider on high-|mkt_excess| days than the single RMSE number below suggests.

**Normality.** The Shapiro-Wilk test on the baseline residuals doesn't reject normality
(p is well above 0.05), consistent with the QQ plot. Nothing here suggests fat tails or
skew badly enough to worry about for this model.

**Independence.** The residual lag-1 check shows a moderate positive autocorrelation. With
only 40 test points, the rough standard error on that estimate is around 0.16, so this sits
close to but not decisively past a typical significance threshold. It's not strong enough to
call a clear violation, but it's exactly the kind of thing that would need re-checking on more
data before being dismissed, especially since this is time-ordered financial data where
autocorrelated residuals are a common failure mode.

**R2 and RMSE.** An R2 around 0.37 means the four factors explain roughly a third of the
variance in the asset's excess return, on out-of-sample data, which is a genuinely useful
amount of explained variance for a single-asset return series (most of the variance in a
return series is close to unexplainable noise by construction). RMSE of about 0.0085 is on the
same order as the daily noise scale the data was generated with, so the model isn't
overfitting the training window.

**Do I trust this model?** For explaining the linear relationship between the four factors
and the asset's return, yes, the coefficients are sane (a market beta near 1, positive size
and momentum loadings, a negative value loading) and the diagnostics mostly hold up. For
precise point prediction, less so: the confirmed heteroscedasticity means a single RMSE
understates risk on high-volatility days, and the borderline autocorrelation is worth watching
rather than ignoring. Next step: a heteroscedasticity-robust standard error (or a model that
lets variance depend on `|mkt_excess|` directly) before trusting any confidence interval built
on this fit.